Total running time: about 45 min.

In [1]:
import os
from glob import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
import cv2
from tqdm import tqdm
import shutil
import xml.etree.ElementTree as ET

## Collect the data and split train set vs test set

In [2]:
# Paths adjusted for notebooks folder structure
PROJECT_ROOT = os.path.join(os.getcwd(), '..')  # Go up one level from notebooks/
INPUT_PATH = os.path.join(PROJECT_ROOT, 'data')
OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'data_processed')
os.makedirs(OUTPUT_PATH, exist_ok=True)
TRAIN_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Train-Annotations-XML', 'DETRAC-Train-Annotations-XML')
TEST_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Test-Annotations-XML', 'DETRAC-Test-Annotations-XML')
ALL_IMAGES_DIR = os.path.join(INPUT_PATH, 'DETRAC-Images', 'DETRAC-Images')

In [3]:
# Split the dataset into training and testing sets based on annotations
train_annotation_files = glob(os.path.join(TRAIN_ANNOTATIONS_DIR, '*.xml'))
test_annotation_files = glob(os.path.join(TEST_ANNOTATIONS_DIR, '*.xml'))
train_image_files = []
test_image_files = []
for ann_file in train_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue
    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    train_image_files.extend(img_files)
for ann_file in test_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue
    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    test_image_files.extend(img_files)

# Copy images to processed directory preserving sequence structure
train_output_dir = os.path.join(OUTPUT_PATH, 'train_images')
test_output_dir = os.path.join(OUTPUT_PATH, 'test_images')
os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

# Copy training images with directory structure
for ann_file in tqdm(train_annotation_files, desc='Copying training sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(train_output_dir, seq_name)
    
    if os.path.isdir(source_dir):
        os.makedirs(dest_dir, exist_ok=True)
        img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))
        for img_file in img_files:
            shutil.copy(img_file, os.path.join(dest_dir, os.path.basename(img_file)))

# Copy testing images with directory structure
for ann_file in tqdm(test_annotation_files, desc='Copying testing sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(test_output_dir, seq_name)
    
    if os.path.isdir(source_dir):
        os.makedirs(dest_dir, exist_ok=True)
        img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))
        for img_file in img_files:
            shutil.copy(img_file, os.path.join(dest_dir, os.path.basename(img_file)))

# Summary
print(f'Total training images: {len(train_image_files)}')
print(f'Total testing images: {len(test_image_files)}')

Found 664 images in MVI_20011
Found 936 images in MVI_20012
Found 437 images in MVI_20032
Found 784 images in MVI_20033
Found 800 images in MVI_20034
Found 800 images in MVI_20035
Found 906 images in MVI_20051
Found 694 images in MVI_20052
Found 800 images in MVI_20061
Found 800 images in MVI_20062
Found 800 images in MVI_20063
Found 800 images in MVI_20064
Found 1200 images in MVI_20065
Found 1660 images in MVI_39761
Found 570 images in MVI_39771
Found 1865 images in MVI_39781
Found 885 images in MVI_39801
Found 1070 images in MVI_39811
Found 880 images in MVI_39821
Found 1420 images in MVI_39851
Found 745 images in MVI_39861
Found 1270 images in MVI_39931
Found 1645 images in MVI_40131
Found 1600 images in MVI_40141
Found 1750 images in MVI_40152
Found 1490 images in MVI_40161
Found 1765 images in MVI_40162
Found 1150 images in MVI_40171
Found 2635 images in MVI_40172
Found 1700 images in MVI_40181
Found 2495 images in MVI_40191
Found 2195 images in MVI_40192
Found 925 images in MVI_

Copying testing sequences: 100%|██████████| 40/40 [02:21<00:00,  3.53s/it]

Total training images: 83791
Total testing images: 56340


## Pre-process the data: create the images and labels folders for both training and testing set

In [4]:
CLASS_MAPPING = {
    'car': 0,
    'bus': 1,
    'van': 2,
}

def process_detrac_annotations(annotations_dir, images_path, labels_path, dataset_type):
    """
    Process DETRAC annotations and create YOLO format labels
    
    Args:
        annotations_dir: Directory containing XML annotation files
        images_path: Output directory for images (with sequence folders)
        labels_path: Output directory for labels (with sequence folders)
        dataset_type: 'train' or 'test'
    """
    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    processed_count = 0
    
    for xml_file in tqdm(os.listdir(annotations_dir), desc=f"Processing {dataset_type} sequences"):
        if not xml_file.endswith('.xml'):
            continue
        
        sequence_name = os.path.splitext(xml_file)[0]
        image_dir = os.path.join(ALL_IMAGES_DIR, sequence_name) 
        
        if not os.path.isdir(image_dir):
            continue

        tree = ET.parse(os.path.join(annotations_dir, xml_file))
        root = tree.getroot()

        img_width, img_height = None, None
        try:
            first_image_path = os.path.join(image_dir, sorted(os.listdir(image_dir))[0])
            with Image.open(first_image_path) as img:
                img_width, img_height = img.size
        except Exception as e:
            continue
        
        # Create sequence-specific directories for images and labels
        seq_images_path = os.path.join(images_path, sequence_name)
        seq_labels_path = os.path.join(labels_path, sequence_name)
        os.makedirs(seq_images_path, exist_ok=True)
        os.makedirs(seq_labels_path, exist_ok=True)
            
        frames = root.findall('frame')
        for frame in frames:
            frame_num = int(frame.get('num'))
            image_filename = f"img{frame_num:05d}.jpg"
            label_filename = f"img{frame_num:05d}.txt"
            
            source_image_path = os.path.join(image_dir, image_filename)
            dest_image_path = os.path.join(seq_images_path, image_filename)
            dest_label_path = os.path.join(seq_labels_path, label_filename)

            if not os.path.exists(source_image_path):
                continue

            shutil.copy(source_image_path, dest_image_path)
            
            yolo_annotations = []
            target_list = frame.find('target_list')
            if target_list is not None:
                for target in target_list.findall('target'):
                    box = target.find('box')
                    attribute = target.find('attribute')
                    
                    vehicle_type = attribute.get('vehicle_type')
                    if vehicle_type not in CLASS_MAPPING:
                        continue
                    
                    class_id = CLASS_MAPPING[vehicle_type]
                    xmin = float(box.get('left'))
                    ymin = float(box.get('top'))
                    width = float(box.get('width'))
                    height = float(box.get('height'))
                    
                    x_center = (xmin + width / 2) / img_width
                    y_center = (ymin + height / 2) / img_height
                    w_norm = width / img_width
                    h_norm = height / img_height
                    
                    yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

            with open(dest_label_path, 'w') as f:
                f.write('\n'.join(yolo_annotations))
            processed_count += 1

    print(f"\n{dataset_type.capitalize()} - Total images and labels successfully processed: {processed_count}")
    return processed_count

# Process training set
train_images_path = os.path.join(OUTPUT_PATH, 'train', 'images')
train_labels_path = os.path.join(OUTPUT_PATH, 'train', 'labels')
train_count = process_detrac_annotations(
    TRAIN_ANNOTATIONS_DIR, 
    train_images_path, 
    train_labels_path, 
    'train'
)

# Process testing set
test_images_path = os.path.join(OUTPUT_PATH, 'test', 'images')
test_labels_path = os.path.join(OUTPUT_PATH, 'test', 'labels')
test_count = process_detrac_annotations(
    TEST_ANNOTATIONS_DIR, 
    test_images_path, 
    test_labels_path, 
    'test'
)

print(f"\n✅ Conversion complete!")
print(f"Training: {train_count} images with labels")
print(f"Testing: {test_count} images with labels")


Processing train sequences: 100%|██████████| 60/60 [05:39<00:00,  5.65s/it]



Train - Total images and labels successfully processed: 82085


Processing test sequences: 100%|██████████| 40/40 [04:59<00:00,  7.50s/it]


Test - Total images and labels successfully processed: 56167

✅ Conversion complete!
Training: 82085 images with labels
Testing: 56167 images with labels
